# Phase 1 — Data Collection & Preprocessing

Join the Fiserv Small Business Index to macro data, align monthly, clean,
and build growth rates and lags.

Output: `data/processed/integrated_monthly.csv`

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

DATA = Path("../data") if Path("../data").exists() else Path("data")
(DATA / "processed").mkdir(exist_ok=True)

## 1. Fiserv

Two exports covering the same panel — one nominal, one deflated. Headers
come out with inconsistent spacing, so clean them on the way in.

In [19]:
def load(filename):
    df = pd.read_csv(DATA / filename)
    df.columns = (df.columns.str.replace(r"\s+", " ", regex=True)
                  .str.replace("Real ", "").str.replace(" - ", "_")
                  .str.replace(" ", "_").str.lower())
    df["date"] = pd.to_datetime(df["period"], format="%Y%m%d")
    return df.rename(columns={"sector_name": "sector", "sub-sector_name": "subsector"})


nom = load("Data_Nominal.csv")
real = load("Data_InflationAdjusted.csv")

print(nom.columns.tolist())
print(f"\n{len(nom):,} rows, {nom.date.nunique()} months, {nom.geo.nunique()} geographies")

['period', 'geo', 'sector', 'subsector', 'sales_index_sa', 'transactional_index_sa', 'sales_mom_%_sa', 'sales_yoy_%_sa', 'transaction_mom_%_sa', 'transaction_yoy_%_sa', 'sales_index_nsa', 'transactional_index_nsa', 'sales_mom_%_nsa', 'sales_yoy_%_nsa', 'transaction_mom_%_nsa', 'transaction_yoy_%_nsa', 'date']

171,828 rows, 92 months, 61 geographies


Check the two files line up before joining anything. The transaction index
appears in both and should match exactly — deflation scales sales, not
transaction counts.

In [20]:
assert len(nom) == len(real)
assert (nom.transactional_index_sa == real.transactional_index_sa).all()
print("ok")

ok


## 2. National series

We want the `US / ALL / ALL` headline. Average ticket is sales divided by
transactions — dollars per basket.

Use the nominal file for this: Fiserv builds the real index by deflating
with CPI, so anything from the nominal/real ratio partly reconstructs CPI
instead of predicting it.

In [21]:
us = nom.query("geo == 'US' and sector == 'ALL' and subsector == 'ALL'").set_index("date")

fiserv = pd.DataFrame({
    "sales_sa": us.sales_index_sa,
    "txn_sa": us.transactional_index_sa,
    "sales_nsa": us.sales_index_nsa,
    "txn_nsa": us.transactional_index_nsa,
}).sort_index()

fiserv["ticket_sa"] = fiserv.sales_sa / fiserv.txn_sa * 100
fiserv["ticket_nsa"] = fiserv.sales_nsa / fiserv.txn_nsa * 100

fiserv = fiserv.add_prefix("fsbi_")
fiserv.tail(3).round(1)

,fsbi_sales_sa,fsbi_txn_sa,fsbi_sales_nsa,fsbi_txn_nsa,fsbi_ticket_sa,fsbi_ticket_nsa
date,,,,,,
2026-06-01,144.9,102.5,151.4,105.3,141.3,143.7
2026-07-01,145.2,102.6,153.7,108.3,141.5,141.9
2026-08-01,144.9,102.4,149.8,107.5,141.5,139.3


In [22]:
# Cross-check against Fiserv's published June 2026 release: index 145,
# sales +2.4% YoY, average ticket +3.7% YoY.
jun = fiserv.loc["2026-06-01"]
yoy = lambda s: np.log(s).diff(12).loc["2026-06-01"] * 100

print(f"index       {jun.fsbi_sales_sa:6.1f}   published 145")
print(f"sales YoY   {yoy(fiserv.fsbi_sales_sa):6.1f}%  published 2.4%")
print(f"ticket YoY  {yoy(fiserv.fsbi_ticket_sa):6.1f}%  published 3.7%")

index        144.9   published 145
sales YoY      2.2%  published 2.4%
ticket YoY     3.6%  published 3.7%


Small gaps are revision — the June release used the July vintage.

## 3. Macro

FRED carries the BLS price indices and Census retail sales as well, so one
source covers everything. Cached after the first pull.

In [23]:
SERIES = {
    "CPIAUCSL": "cpi_sa", "CPIAUCNS": "cpi_nsa", "CPILFESL": "core_cpi_sa",
    "PCEPI": "pce_price", "PCEPILFE": "core_pce_price",

    "CUSR0000SEFV": "cpi_food_away", "CUSR0000SETB01": "cpi_gasoline",
    "CUSR0000SAF11": "cpi_food_home", "CUSR0000SAH1": "cpi_shelter",
    "CPIAPPSL": "cpi_apparel", "CUSR0000SETA02": "cpi_used_cars",

    "RSAFS": "retail_sa", "RSAFSNA": "retail_nsa", "RSFSXMV": "retail_ex_auto",
    "PCE": "pce", "PCEC96": "real_pce", "PCEDG": "pce_durables",
    "PCEND": "pce_nondurables", "PCES": "pce_services",
    "DSPIC96": "real_income", "PSAVERT": "savings_rate",

    "UNRATE": "unemployment", "PAYEMS": "payrolls", "FEDFUNDS": "fed_funds",
    "DGS2": "treasury_2y", "DGS10": "treasury_10y", "T10YIE": "breakeven_10y",
    "UMCSENT": "sentiment", "MICH": "inflation_expect",
    "VIXCLS": "vix", "DTWEXBGS": "dollar", "DCOILWTICO": "oil",
}

In [24]:
cache = DATA / "macro_raw.csv"

if cache.exists():
    macro_raw = pd.read_csv(cache, index_col=0, parse_dates=True)
else:
    from pandas_datareader import data as web
    macro_raw = web.DataReader(list(SERIES), "fred", "2015-01-01")
    macro_raw.columns = [SERIES[c] for c in macro_raw.columns]
    macro_raw.to_csv(cache)

print(f"{macro_raw.shape[1]} series, {macro_raw.index.min():%Y-%m} to {macro_raw.index.max():%Y-%m}")

32 series, 2015-01 to 2026-09


### 3.1 New Macro Data 



Included new macro data series from FRED, influeced by inflation prediction research papers. All of them are monthly with some of them be averaged. These series include:

Market Yield on U.S. Treasury Securities at 5-Year Constant Maturity, Quoted on an Investment Basis , Monthly Average : DGS5

Market Yield on U.S. Treasury Securities at 3-Month Constant Maturity, Quoted on an Investment Basis, Monthly Average : DGS3MO

Spot Crude Oil Price: West Texas Intermediate: WTISPLC

University of Michigan: Consumer Sentiment: UMCSENT

Industrial Production: Total Index  : INDPRO

All Employees, Total Nonfarm : PAYEMS

New Privately-Owned Housing Units Started: Total Units : HOUST

Treasury 5 Year - Treasury 3 Month: dgs5_minus_dgs3mo

In [25]:
cache2 = DATA / "new_macro_data.csv"
if cache2.exists():
    new_macro_data = pd.read_csv(cache2, index_col=0, parse_dates=True)
else:
    from pandas_datareader import data as web
    new_macro_data = web.DataReader(list(SERIES), "fred", "2024-01-01")
    new_macro_data.columns = [SERIES[c] for c in new_macro_data.columns]
    new_macro_data.to_csv(cache2)

print(f"{new_macro_data.shape[1]} series, {new_macro_data.index.min():%Y-%m} to {new_macro_data.index.max():%Y-%m}")


8 series, 2010-01 to 2026-08


## 4. Align to monthly

Yields, VIX, the dollar and oil are daily. Fiserv stamps periods at month
start, so everything has to match — an off-by-one month here would be
invisible and would wreck everything downstream.

In [26]:
macro = macro_raw.resample("MS").last()

assert (macro.index.day == 1).all() and (fiserv.index.day == 1).all()
print(f"{macro.shape[0]} months")

141 months


In [27]:
macro

,cpi_sa,cpi_nsa,core_cpi_sa,pce_price,core_pce_price,cpi_food_away,cpi_gasoline,cpi_food_home,cpi_shelter,cpi_apparel,cpi_used_cars,retail_sa,retail_nsa,retail_ex_auto,pce,real_pce,pce_durables,pce_nondurables,pce_services,real_income,savings_rate,unemployment,payrolls,fed_funds,treasury_2y,treasury_10y,breakeven_10y,sentiment,inflation_expect,vix,dollar,oil
DATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-01,234.747,233.707,239.811,96.654,96.214,253.037,199.553,242.553,274.762,126.129,144.875,428208.0,391738.0,341152.0,12066.7,12484.7,1280.2,2587.0,8199.5,13797.7,6.3,5.7,140568.0,0.11,0.47,1.68,1.65,98.1,2.5,20.97,105.5884,47.79
2015-02-01,235.342,234.722,240.172,96.825,96.324,253.719,206.901,242.598,275.470,125.960,147.264,427119.0,381513.0,341440.0,12116.6,12514.1,1283.8,2604.6,8228.3,13848.0,6.4,5.5,140827.0,0.11,0.63,2.00,1.83,95.4,2.8,13.34,105.5981,49.84
2015-03-01,235.976,236.119,240.755,97.008,96.470,254.108,214.684,241.633,276.258,126.626,147.694,433647.0,438118.0,343998.0,12176.1,12551.8,1309.2,2625.7,8241.2,13811.3,5.9,5.4,140923.0,0.11,0.56,1.94,1.76,93.0,3.0,15.29,107.4396,47.72
2015-04-01,236.222,236.599,241.346,97.094,96.648,254.727,211.577,241.106,277.003,126.390,148.404,434470.0,431604.0,344264.0,12209.1,12574.8,1315.9,2617.4,8275.8,13842.0,5.9,5.4,141196.0,0.12,0.58,2.05,1.94,95.9,2.6,14.55,105.3120,59.62
2015-05-01,237.001,237.805,241.688,97.327,96.766,255.322,225.281,241.107,277.524,125.831,149.290,437865.0,456436.0,346941.0,12275.4,12612.7,1319.6,2649.4,8306.5,13865.6,5.8,5.6,141538.0,0.12,0.61,2.12,1.80,90.7,2.8,13.84,106.7388,60.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-01,333.979,335.123,336.121,131.576,130.147,394.728,378.660,320.836,427.998,137.069,180.005,766192.0,796140.0,625517.0,22136.8,16825.5,2386.0,4515.7,15235.1,18000.1,2.8,4.3,158861.0,3.63,3.98,4.45,2.38,44.8,4.8,15.32,118.8783,91.16
2026-06-01,332.568,333.952,336.065,131.454,130.338,395.633,341.980,321.446,428.501,136.313,179.591,768587.0,776516.0,624447.0,22214.2,16900.0,2404.7,4494.6,15314.9,18054.9,2.6,4.2,158892.0,3.63,4.14,4.44,2.24,49.5,4.6,16.45,120.9248,70.56
2026-07-01,332.813,333.918,336.789,131.659,130.658,396.859,332.215,321.216,429.095,136.434,180.306,764462.0,784603.0,622897.0,22250.4,16901.3,2380.1,4469.3,15401.0,18122.5,3.0,4.1,158913.0,3.63,4.28,4.75,2.28,55.2,4.2,15.99,119.7034,86.16


### 4.1 Merged with new macro data

In [28]:
macro = macro.merge(new_macro_data, left_on="DATE",right_on="DATE", how="left")
macro

,cpi_sa,cpi_nsa,core_cpi_sa,pce_price,core_pce_price,cpi_food_away,cpi_gasoline,cpi_food_home,cpi_shelter,cpi_apparel,cpi_used_cars,retail_sa,retail_nsa,retail_ex_auto,pce,real_pce,pce_durables,pce_nondurables,pce_services,real_income,savings_rate,unemployment,payrolls,fed_funds,treasury_2y,treasury_10y,breakeven_10y,sentiment,inflation_expect,vix,dollar,oil,dgs3mo,dgs5,houst,indpro,payems,wtisplc,umcsent,dgs5_minus_dgs3mo
DATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-01,234.747,233.707,239.811,96.654,96.214,253.037,199.553,242.553,274.762,126.129,144.875,428208.0,391738.0,341152.0,12066.7,12484.7,1280.2,2587.0,8199.5,13797.7,6.3,5.7,140568.0,0.11,0.47,1.68,1.65,98.1,2.5,20.97,105.5884,47.79,0.03,1.37,1085.0,102.8905,140568.0,47.22,98.1,1.34
2015-02-01,235.342,234.722,240.172,96.825,96.324,253.719,206.901,242.598,275.470,125.960,147.264,427119.0,381513.0,341440.0,12116.6,12514.1,1283.8,2604.6,8228.3,13848.0,6.4,5.5,140827.0,0.11,0.63,2.00,1.83,95.4,2.8,13.34,105.5981,49.84,0.02,1.47,886.0,102.2335,140827.0,50.58,95.4,1.45
2015-03-01,235.976,236.119,240.755,97.008,96.470,254.108,214.684,241.633,276.258,126.626,147.694,433647.0,438118.0,343998.0,12176.1,12551.8,1309.2,2625.7,8241.2,13811.3,5.9,5.4,140923.0,0.11,0.56,1.94,1.76,93.0,3.0,15.29,107.4396,47.72,0.03,1.52,960.0,101.8914,140923.0,47.82,93.0,1.49
2015-04-01,236.222,236.599,241.346,97.094,96.648,254.727,211.577,241.106,277.003,126.390,148.404,434470.0,431604.0,344264.0,12209.1,12574.8,1315.9,2617.4,8275.8,13842.0,5.9,5.4,141196.0,0.12,0.58,2.05,1.94,95.9,2.6,14.55,105.3120,59.62,0.02,1.35,1190.0,101.3355,141196.0,54.45,95.9,1.33
2015-05-01,237.001,237.805,241.688,97.327,96.766,255.322,225.281,241.107,277.524,125.831,149.290,437865.0,456436.0,346941.0,12275.4,12612.7,1319.6,2649.4,8306.5,13865.6,5.8,5.6,141538.0,0.12,0.61,2.12,1.80,90.7,2.8,13.84,106.7388,60.25,0.02,1.54,1079.0,100.8883,141538.0,59.27,90.7,1.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-01,333.979,335.123,336.121,131.576,130.147,394.728,378.660,320.836,427.998,137.069,180.005,766192.0,796140.0,625517.0,22136.8,16825.5,2386.0,4515.7,15235.1,18000.1,2.8,4.3,158861.0,3.63,3.98,4.45,2.38,44.8,4.8,15.32,118.8783,91.16,3.69,4.15,1182.0,102.6396,158861.0,102.13,44.8,0.46
2026-06-01,332.568,333.952,336.065,131.454,130.338,395.633,341.980,321.446,428.501,136.313,179.591,768587.0,776516.0,624447.0,22214.2,16900.0,2404.7,4494.6,15314.9,18054.9,2.6,4.2,158892.0,3.63,4.14,4.44,2.24,49.5,4.6,16.45,120.9248,70.56,3.81,4.21,1439.0,102.8420,158892.0,84.81,49.5,0.40
2026-07-01,332.813,333.918,336.789,131.659,130.658,396.859,332.215,321.216,429.095,136.434,180.306,764462.0,784603.0,622897.0,22250.4,16901.3,2380.1,4469.3,15401.0,18122.5,3.0,4.1,158913.0,3.63,4.28,4.75,2.28,55.2,4.2,15.99,119.7034,86.16,3.87,4.33,1309.0,103.0454,158913.0,80.46,55.2,0.46


Series end in different months because of the release calendar — CPI and
retail sales for month *m* land mid *m+1*, PCE at the end of it. So PCE
always trails by a month.

In [29]:
macro.apply(lambda s: s.last_valid_index()).value_counts().sort_index()

2026-07-01    12
2026-08-01    22
2026-09-01     6
Name: count, dtype: int64

## 5. Missing values

Gaps inside a series are worth filling; gaps at the end are just data that
hasn't been published yet. `limit_area="inside"` keeps interpolation from
running past the last real observation.

In [30]:
interior = macro.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum())
print(interior[interior > 0] if interior.any() else "no interior gaps")

macro = macro.interpolate(limit=2, limit_area="inside")

cpi_sa           1
cpi_nsa          1
core_cpi_sa      1
cpi_food_away    1
cpi_food_home    1
cpi_shelter      1
cpi_apparel      1
unemployment     1
dtype: int64


## 6. Merge

Inner join on month start. Fiserv starts in 2019, so that binds — about
90 months, a small sample that should keep the feature count down later.

In [31]:
df = fiserv.join(macro, how="inner")
print(f"{df.shape[0]} months x {df.shape[1]} columns, {df.index.min():%Y-%m} to {df.index.max():%Y-%m}")

92 months x 46 columns, 2019-01 to 2026-08


## 7. Features

Log growth for levels, plain differences for rates — 2% to 3% is a
percentage-point move, and a log ratio breaks on negative values.

In [32]:
RATES = ["fed_funds", "treasury_2y", "treasury_10y", "breakeven_10y",
         "savings_rate", "unemployment", "inflation_expect", "vix","dgs3mo","dgs5","dgs5_minus_dgs3mo"]

levels = df.drop(columns=RATES)
rates = df[RATES]

growth = pd.concat([
    (np.log(levels).diff() * 100).add_suffix("_mom"),
    (np.log(levels).diff(12) * 100).add_suffix("_yoy"),
    rates.diff().add_suffix("_mom"),
    rates.diff(12).add_suffix("_yoy"),
], axis=1)

print(f"{growth.shape[1]} growth columns")

92 growth columns


In [33]:
smooth = ["fsbi_sales_sa_yoy", "fsbi_ticket_sa_yoy", "fsbi_txn_sa_yoy",
          "retail_sa_yoy", "cpi_sa_yoy"]

ma = pd.concat([growth[smooth].rolling(w).mean().add_suffix(f"_ma{w}")
                for w in (3, 6)], axis=1)
ma.tail(3).round(2)

,fsbi_sales_sa_yoy_ma3,fsbi_ticket_sa_yoy_ma3,fsbi_txn_sa_yoy_ma3,retail_sa_yoy_ma3,cpi_sa_yoy_ma3,fsbi_sales_sa_yoy_ma6,fsbi_ticket_sa_yoy_ma6,fsbi_txn_sa_yoy_ma6,retail_sa_yoy_ma6,cpi_sa_yoy_ma6
date,,,,,,,,,,
2026-06-01,1.30,3.25,-1.95,6.18,3.73,1.14,2.90,-1.76,5.00,3.20
2026-07-01,1.46,3.27,-1.82,6.16,3.58,1.28,2.95,-1.67,5.28,3.35
2026-08-01,1.67,3.25,-1.58,5.78,3.32,1.30,3.10,-1.80,5.57,3.50


### Lags

Lag *k* means predicting month *t* from data at *t−k*. Which lag is honest
depends on the calendar: Fiserv publishes month *m* on the 2nd of *m+1*,
CPI lands around the 13th. So for CPI, lag 0 is already public — a
**nowcast**. Lag 1 is a real one-month-ahead **forecast**.

Both are fine. Mixing them without saying so is how backtests flatter
themselves.

In [34]:
signals = [c for c in growth.columns if c.startswith("fsbi_")]

lags = pd.concat([growth[signals].shift(k).add_suffix(f"_lag{k}")
                  for k in (1, 2, 3, 6, 12)], axis=1)

integrated = pd.concat([df, growth, ma, lags], axis=1)
print(f"{integrated.shape[1]} columns, {integrated.dropna().shape[0]} complete rows")

208 columns, 67 complete rows


## 8. Scaling

Not applied to the saved file. Fitting a scaler on everything leaks
test-period means and standard deviations into training, so it has to be
refit inside each walk-forward fold.

In [35]:
from sklearn.preprocessing import StandardScaler

X = integrated[["fsbi_ticket_sa_yoy", "fsbi_txn_sa_yoy"]].dropna()
train, test = X.iloc[:-12], X.iloc[-12:]

scaler = StandardScaler().fit(train)
print(f"train mean {scaler.transform(train).mean():+.2f}")
print(f"test mean  {scaler.transform(test).mean():+.2f}   <- not zero, which is correct")

train mean +0.00
test mean  -0.44   <- not zero, which is correct


## 9. Save

In [36]:
integrated.to_csv(DATA / "processed/integrated_monthly.csv")

pd.DataFrame({
    "column": integrated.columns,
    "n_obs": integrated.notna().sum().values,
    "first": integrated.apply(lambda s: s.first_valid_index()).values,
}).to_csv(DATA / "processed/data_dictionary.csv", index=False)

print(f"saved {integrated.shape[0]} months x {integrated.shape[1]} columns")

saved 92 months x 208 columns


## Notes for later phases

- Nominal, not real — the real index is deflated with CPI, so using it to
  predict CPI is circular.
- SA and NSA both kept. Nearly identical YoY, very different MoM. NSA is
  the right basis for anything traded, since CPI settles unadjusted.
- Saved unscaled; scale inside CV folds.
- Lag 0 is a nowcast, lag 1 is a forecast — keep that distinction in the
  backtest.
- ~90 months, mostly COVID and the inflation spike. Start with few features.